# Data Processing

Loads all `eval_results.csv` files from a list of top-level experiment folders and merges them into two DataFrames:
- **`sem_var_df`**: one row per semantic variation (aggregated across shuffles), from `eval_results.csv`
- **`replicate_df`**: one row per shuffle/replicate (raw JSONL files)

Run `eval_all.sh` first to generate the `eval_results.csv` files, then run this notebook.

## Setup

In [1]:
import os
os.chdir("../..")

In [2]:
import re
import pandas as pd
import numpy as np
from glob import glob

## Configuration

Add top-level experiment folder paths here. Each entry is a folder containing `model_name_*` subfolders.

In [3]:
toplevel_paths = [
    # Farm domain
    "experiments/20260118_farm_novar",
    "experiments/20260124_farm_4sd_var",
    "experiments/20260124_farm_2sd_var",
    # Abstract Bandit domain
    "experiments/20260120_ab_novar",
    "experiments/20260124_abandit_4sd_var",
    "experiments/20260124_abandit_2sd_var",
    # Clothing Recommendation domain
    "experiments/20260312_rec_novar",
    "experiments/20260312_rec_4sd_var",
    "experiments/20260312_rec_2sd_var",
    # Gemini models
    "experiments/20260311_gemini_farm_novar",
    "experiments/20260312_gemini_ab_novar",
    # Explicit debiasing with explore instruction (warn prompt)
    "experiments/20260523_ab_warn_novar",
    "experiments/20260523_farm_warn_novar",
    "experiments/20260523_rec_warn_novar",
    # Explicit debiasing without explore instruction (warn, no-explore prompt)
    "experiments/20260524_ab_warn_noexp_novar",
    "experiments/20260524_farm_warn_noexp_novar",
    "experiments/20260524_rec_warn_noexp_novar",
]

## Path Parsing Functions

Auto-detect experimental variables from folder path names:
- **Domain**: from top-level folder name (`farm` → Farm, `ab`/`abandit` → Bandit, `rec` → Clothing Recommendation)
- **Variance**: from top-level folder name (`4sd_var` → Low, `2sd_var` → High, else → No)
- **Model**: from `model_name_[Model]` subfolder name
- **History**: from `prompt_fullhist`/`prompt_summhist` in subfolder name (default: Summarized)

In [4]:
def parse_toplevel_path(toplevel_path):
    """Extract domain and variance from a top-level experiment folder path."""
    name = os.path.basename(toplevel_path.rstrip("/"))

    # Domain
    if "rec" in name:
        domain = "Clothing Recommendation"
    elif "farm" in name:
        domain = "Farm"
    elif "abandit" in name or "ab" in name:
        domain = "Bandit"
    else:
        domain = "Unknown"

    # Variance (4sd = means are far apart = Low noise; 2sd = means closer = High noise)
    if "4sd_var" in name:
        variance = "Low"
    elif "2sd_var" in name:
        variance = "High"
    else:
        variance = "No"

    # Explicit debiasing: warn experiments have "warn" in the folder name
    explicit_debiasing = "warn" in name

    # Explore instruction: warn experiments that do NOT have "noexp" in the folder name
    explore_instruction = "warn" in name and "noexp" not in name

    return domain, variance, explicit_debiasing, explore_instruction


def parse_superfolder_name(subfolder_name):
    """Extract model and history from a model_name_* subfolder name."""
    model_match = re.search(r"model_name_(.+?)(?:_prompt_|$)", subfolder_name)
    model = model_match.group(1) if model_match else "Unknown"

    if "prompt_fullhist" in subfolder_name:
        history = "Full"
    elif "prompt_summhist" in subfolder_name:
        history = "Summarized"
    else:
        history = "Summarized"  # default when not specified

    return model, history


def fix_legacy_nomenclature(nomenclature, domain):
    """Map old sem_rel_ nomenclature to the current scheme per domain."""
    if "sem_rel" not in nomenclature:
        return nomenclature
    if domain == "Farm":
        return nomenclature.replace("sem_rel", "world")
    elif domain == "Bandit":
        return nomenclature.replace("sem_rel", "ordinal")
    return nomenclature

## Discover Superfolders

Walk each top-level path and collect all `model_name_*` subfolders with their metadata.

In [5]:
records = []
for toplevel_path in toplevel_paths:
    if not os.path.isdir(toplevel_path):
        print(f"Warning: {toplevel_path} does not exist — skipping.")
        continue
    domain, variance, explicit_debiasing, explore_instruction = parse_toplevel_path(toplevel_path)

    for entry in sorted(os.listdir(toplevel_path)):
        if not entry.startswith("model_name_"):
            continue
        superfolder_path = os.path.join(toplevel_path, entry)
        if not os.path.isdir(superfolder_path):
            continue
        model, history = parse_superfolder_name(entry)
        records.append({
            "toplevel_path": toplevel_path,
            "superfolder_path": superfolder_path,
            "domain": domain,
            "model": model,
            "history": history,
            "variance": variance,
            "explicit_debiasing": explicit_debiasing,
            "explore_instruction": explore_instruction,
        })

superfolders_df = pd.DataFrame(records)
print(f"Found {len(superfolders_df)} superfolders.")
superfolders_df

Found 91 superfolders.


,toplevel_path,superfolder_path,domain,model,history,variance,explicit_debiasing,explore_instruction
0,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False
1,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Summarized,No,False,False
2,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Olm...,Farm,Olmo-3.1-32B-Instruct,Summarized,No,False,False
3,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Qwe...,Farm,Qwen3-14B,Summarized,No,False,False
4,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Qwe...,Farm,Qwen3-14B,Full,No,False,False
...,...,...,...,...,...,...,...,...
86,experiments/20260524_farm_warn_noexp_novar,experiments/20260524_farm_warn_noexp_novar/mod...,Farm,Qwen3-32B,Summarized,No,True,False
87,experiments/20260524_farm_warn_noexp_novar,experiments/20260524_farm_warn_noexp_novar/mod...,Farm,gemini-3.1-flash-lite-preview,Summarized,No,True,False
88,experiments/20260524_rec_warn_noexp_novar,experiments/20260524_rec_warn_noexp_novar/mode...,Clothing Recommendation,Olmo-3.1-32B-Instruct,Summarized,No,True,False
89,experiments/20260524_rec_warn_noexp_novar,experiments/20260524_rec_warn_noexp_novar/mode...,Clothing Recommendation,Qwen3-32B,Summarized,No,True,False


## Per-Semantic-Variation DataFrame

Loads `eval_results.csv` from each superfolder (one row per semantic variation, metrics averaged across shuffles).

In [6]:
sem_var_dfs = []
for _, row in superfolders_df.iterrows():
    csv_path = os.path.join(row["superfolder_path"], "eval_results.csv")
    if not os.path.exists(csv_path):
        print(f"No eval_results.csv at {csv_path} — skipping. (Run eval_all.sh first.)")
        continue
    df = pd.read_csv(csv_path)
    df["domain"] = row["domain"]
    df["model"] = row["model"]
    df["history"] = row["history"]
    df["variance"] = row["variance"]
    df["superfolder_path"] = row["superfolder_path"]
    df["explicit_debiasing"] = row["explicit_debiasing"]
    df["explore_instruction"] = row["explore_instruction"]
    df["nomenclature"] = df["nomenclature"].apply(
        lambda n: fix_legacy_nomenclature(n, row["domain"])
    )
    sem_var_dfs.append(df)

if sem_var_dfs:
    sem_var_df = pd.concat(sem_var_dfs, ignore_index=True)
    print(f"sem_var_df shape: {sem_var_df.shape}")
    sem_var_df.head()
else:
    print("No eval_results.csv files found. Run eval_all.sh first.")
    sem_var_df = pd.DataFrame()

sem_var_df shape: (1859, 171)


## Classical Baselines (UCB1, TS)

Loads `eval_results.csv` files produced by `run_baselines.py` and appends them to `sem_var_df`.
Run `python run_baselines.py` from the repo root first.

Folder structure:
```
experiments/20260326_classicalsolver/
  {method}/
    {scale}/
      eval_results.csv   ← 3 rows, one per noise level (variance = No / Low / High)
```

In [7]:
BASELINE_PATH = "experiments/20260326_classicalsolver"
baseline_dfs = []

if os.path.isdir(BASELINE_PATH):
    for method in sorted(os.listdir(BASELINE_PATH)):
        method_path = os.path.join(BASELINE_PATH, method)
        if not os.path.isdir(method_path):
            continue
        for scale_folder in sorted(os.listdir(method_path)):
            csv_path = os.path.join(method_path, scale_folder, "eval_results.csv")
            if not os.path.exists(csv_path):
                continue
            df = pd.read_csv(csv_path)
            # variance is already in the CSV (No / Low / High, mapped from sigma)
            df["domain"] = "Classical"
            df["model"] = method           # UCB1 or TS
            df["history"] = "Symbolic"     # no prompt history used
            df["explicit_debiasing"] = False
            df["explore_instruction"] = False
            df["superfolder_path"] = method_path
            baseline_dfs.append(df)

if baseline_dfs:
    baseline_df = pd.concat(baseline_dfs, ignore_index=True)
    sem_var_df = pd.concat([sem_var_df, baseline_df], ignore_index=True)
    print(f"Appended {len(baseline_df)} classical baseline rows. sem_var_df shape: {sem_var_df.shape}")
else:
    print(f"No classical baseline results at {BASELINE_PATH}. Run `python run_baselines.py` first.")

Appended 12 classical baseline rows. sem_var_df shape: (1871, 172)


In [8]:
sem_var_df[["domain", "model", "history", "variance", "nomenclature", "scale", "n_shuffles"]].value_counts().reset_index(name="count")

,domain,model,history,variance,nomenclature,scale,n_shuffles,count
0,Bandit,Qwen3-32B,Summarized,No,ordinal_helpful,high_neg_scale,10,3
1,Farm,Olmo-3.1-32B-Instruct,Summarized,No,ordinal_helpful,high_neg_scale,10,3
2,Bandit,Qwen3-32B,Summarized,No,ordinal_mislead,high_scale,10,3
3,Bandit,Olmo-3.1-32B-Instruct,Summarized,No,alphanumeric,high_scale,10,3
4,Bandit,Qwen3-32B,Summarized,No,sent_new_helpful,high_neg_scale,10,3
...,...,...,...,...,...,...,...,...
1562,Clothing Recommendation,Llama3-8B,Summarized,Low,world_mislead,high_neg_scale,10,1
1563,Clothing Recommendation,Llama3-8B,Summarized,Low,world_helpful,low_scale,10,1
1564,Clothing Recommendation,Llama3-8B,Summarized,Low,world_helpful,low_neg_scale,10,1
1565,Clothing Recommendation,Llama3-8B,Summarized,Low,world_helpful,high_scale,10,1


In [9]:
sem_var_df.to_csv("experiments/merged_sem_var_results.csv", index=False)
print("Saved experiments/merged_sem_var_results.csv")

Saved experiments/merged_sem_var_results.csv


## Per-Replicate DataFrame

Walks each superfolder's shuffle directories and finds the most recent valid JSONL file (≥10 rows) per shuffle. One row per shuffle/replicate.

In [10]:
# Nomenclatures sorted longest-first to avoid partial substring matches
NOMENCLATURES = sorted([
    "alphanumeric",
    "sem_rel_helpful", "sem_rel_mislead",
    "sent_new_helpful", "sent_new_mislead",
    "sent_helpful", "sent_mislead",
    "ordinal_helpful", "ordinal_mislead",
    "world_helpful", "world_mislead",
    "baseline",          # classical solver baselines (UCB1, TS)
], key=len, reverse=True)

SCALES = ["high_neg_scale", "low_neg_scale", "high_scale", "low_scale"]


def find_valid_jsonl(shuffle_path, min_rows=10):
    """Return the most recent JSONL in shuffle_path with at least min_rows lines."""
    candidates = sorted(glob(f"{shuffle_path}/*.jsonl"), key=os.path.getmtime, reverse=True)
    for path in candidates:
        try:
            df = pd.read_json(path, lines=True)
            if len(df) >= min_rows:
                return path
        except Exception:
            continue
    return None


replicate_records = []

for _, sf_row in superfolders_df.iterrows():
    superfolder_path = sf_row["superfolder_path"]

    for sem_var_folder in sorted(os.listdir(superfolder_path)):
        sem_var_path = os.path.join(superfolder_path, sem_var_folder)
        if not os.path.isdir(sem_var_path):
            continue

        shuffle_folder = os.path.join(sem_var_path, "shuffles")
        if not os.path.isdir(shuffle_folder):
            continue

        # Identify nomenclature and scale from folder name
        nom_matches = [n for n in NOMENCLATURES if n in sem_var_folder]
        scale_matches = [s for s in SCALES if s in sem_var_folder]
        if len(nom_matches) != 1 or len(scale_matches) != 1:
            continue

        nomenclature = fix_legacy_nomenclature(nom_matches[0], sf_row["domain"])
        scale = scale_matches[0]

        for shuffle_path in sorted(glob(f"{shuffle_folder}/shuffle_*")):
            jsonl_file = find_valid_jsonl(shuffle_path)
            if jsonl_file is None:
                continue
            shuffle_num = int(os.path.basename(shuffle_path).split("_")[-1])
            replicate_records.append({
                "toplevel_path": sf_row["toplevel_path"],
                "superfolder_path": superfolder_path,
                "domain": sf_row["domain"],
                "model": sf_row["model"],
                "history": sf_row["history"],
                "variance": sf_row["variance"],
                "explicit_debiasing": sf_row["explicit_debiasing"],
                "explore_instruction": sf_row["explore_instruction"],
                "nomenclature": nomenclature,
                "scale": scale,
                "shuffle_number": shuffle_num,
                "jsonl_file": jsonl_file,
            })

replicate_df = pd.DataFrame(replicate_records)
print(f"replicate_df shape: {replicate_df.shape}")
replicate_df.head()

replicate_df shape: (18534, 12)


,toplevel_path,superfolder_path,domain,model,history,variance,explicit_debiasing,explore_instruction,nomenclature,scale,shuffle_number,jsonl_file
0,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False,alphanumeric,high_neg_scale,0,experiments/20260118_farm_novar/model_name_Lla...
1,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False,alphanumeric,high_neg_scale,1,experiments/20260118_farm_novar/model_name_Lla...
2,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False,alphanumeric,high_neg_scale,2,experiments/20260118_farm_novar/model_name_Lla...
3,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False,alphanumeric,high_neg_scale,3,experiments/20260118_farm_novar/model_name_Lla...
4,experiments/20260118_farm_novar,experiments/20260118_farm_novar/model_name_Lla...,Farm,Llama3-8B,Full,No,False,False,alphanumeric,high_neg_scale,4,experiments/20260118_farm_novar/model_name_Lla...


In [11]:
replicate_df[["domain", "model", "history", "variance"]].value_counts().reset_index(name="count")

,domain,model,history,variance,count
0,Farm,Qwen3-32B,Summarized,No,779
1,Farm,Olmo-3.1-32B-Instruct,Summarized,No,699
2,Farm,gemini-3.1-flash-lite-preview,Summarized,No,679
3,Clothing Recommendation,Olmo-3.1-32B-Instruct,Summarized,No,660
4,Clothing Recommendation,Qwen3-32B,Summarized,No,659
5,Clothing Recommendation,gemini-3.1-flash-lite-preview,Summarized,No,649
6,Bandit,Qwen3-32B,Summarized,No,580
7,Bandit,Olmo-3.1-32B-Instruct,Summarized,No,500
8,Bandit,gemini-3.1-flash-lite-preview,Summarized,No,489
9,Farm,Qwen3-14B,Summarized,Low,360


In [12]:
replicate_df.to_csv("experiments/merged_replicate_results.csv", index=False)
print("Saved experiments/merged_replicate_results.csv")

Saved experiments/merged_replicate_results.csv
